# Inspect right-arm joint actions

Plots the **commanded** right-arm targets (`cmd_right_j1..j7`, `cmd_grip_right`)
against time from a joint-tracking CSV written by
`camelo.runner.joint_log.JointTrackingLog` (see `camelo/runner/joint_log.py`
for the authoritative column reference).

## What the CSV means

One row per control tick, fixed columns:

```
t, t_sim, tick, phase,
cmd_left_j1..j7, cmd_right_j1..j7,
meas_left_j1..j7, meas_right_j1..j7,
cmd_grip_left, cmd_grip_right, meas_grip_left, meas_grip_right,
clamped, infer_ms, max_publish_gap_s, state_age_s
```

- **`t`** — seconds since the log opened (monotonic clock, immune to scene
  resets). **`t_sim`** — the same tick's `Obs.t_sim`, the clock the executor
  interpolates chunks on; join other per-tick CSVs on this column, not `t`.
- **`cmd_*`** — the executor's `Command`, read *after* the grasp-gate
  override and the per-tick delta clamp, in the ROBOT frame. This is what
  the policy asked the arm to do, not necessarily the wire value: under
  `--arm-command-frame gello` the wire carries `g0 + dir*(q - q0)`, and a
  side left out of `--arms` republishes a frozen pose.
- **`meas_*`** — the measured joint state fed back from the arm at that
  tick.
- **`cmd_grip_left/right`, `meas_grip_left/right`** — gripper open
  fractions (`1.0` = open), commanded and measured.
- **`phase`** — written by the caller; the T5 tracking summary is computed
  only over `phase == "rollout"` rows.
- **`clamped`** — `1`/`0` when the executor's clamp was evaluated that
  tick, **empty** when it wasn't reached.
- **`infer_ms`** — policy inference time for the tick, when available.
- **`max_publish_gap_s`** — worst gap seen so far between consecutive arm
  publishes.
- **`state_age_s`** — worst WALL age, at that tick, of the `JointState`
  streams `meas_*` is read from; a dead stream shows up here even if
  `meas_*` still looks plausible (a frozen sample repeated).

**Empty is not zero.** A tick where the executor had nothing to send (no
chunk yet, a clock rebase) still gets a row, but every `cmd_*` field and
`clamped` are left empty — never `0.0` — so "no command" can never be
misread as "commanded zero". This notebook drops those empty ticks per
column before plotting, so a line has a gap wherever nothing was
commanded rather than dipping to zero.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Path to a joint-tracking CSV (see markdown above for the column format).
# Edit this, or set the JOINT_TRACE_CSV env var and re-run all cells (e.g.
# from `jupyter nbconvert --execute` / papermill) to point at a different
# rig/eval run without touching the notebook.
ROOT = Path("/Users/maximilianbomer/workspace/camelo-ebim")
CSV_PATH: Path = Path(
    os.environ.get("JOINT_TRACE_CSV", str(ROOT / "outputs/rig/t6v_v01_143309.csv"))
    # os.environ.get("JOINT_TRACE_CSV", str(ROOT / "outputs/rig/t6a_a03_150950.csv"))
)
CSV_PATH

In [ ]:
df = pd.read_csv(CSV_PATH)

RIGHT_ACTION_COLUMNS = [f"cmd_right_j{i}" for i in range(1, 8)] + ["cmd_grip_right"]
missing = [c for c in RIGHT_ACTION_COLUMNS if c not in df.columns]
if missing:
    raise ValueError(f"{CSV_PATH} is missing expected columns: {missing}")

print(f"{len(df)} rows, phases={sorted(df['phase'].unique())}")
df[["t", *RIGHT_ACTION_COLUMNS]].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for col in RIGHT_ACTION_COLUMNS:
    sub = df[["t", col]].dropna()
    ax.plot(sub["t"], sub[col], marker=".", markersize=3, linewidth=1, label=col)

ax.set_xlabel("t (s, since rollout start)")
ax.set_ylabel("action value")
ax.set_title(f"Right-arm commanded actions vs time — {CSV_PATH.name}")
ax.legend(loc="upper right", ncol=2, fontsize=8)
ax.grid(True, alpha=0.3)
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for col in RIGHT_ACTION_COLUMNS:
    sub = df[["t", col]].dropna()
    ax.plot(sub["t"], sub[col], marker=".", markersize=3, linewidth=1, label=col)

ax.set_xlabel("t (s, since rollout start)")
ax.set_ylabel("action value")
ax.set_title(f"Right-arm commanded actions vs time — {CSV_PATH.name}")
ax.legend(loc="upper right", ncol=2, fontsize=8)
ax.grid(True, alpha=0.3)
fig.tight_layout()